# CNN-DAE — Treino
**Execute as células em ordem.**

Antes de começar: **Runtime → Change runtime type → T4 GPU**.

---
## 1 · Google Drive
Checkpoints e logs salvos no Drive. Se a sessão cair, o treino pode ser retomado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cnn_dae'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs',        exist_ok=True)
print('Drive montado. Pasta do projeto:', DRIVE_DIR)

---
## 2 · Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU encontrada:', result.stdout.strip())
else:
    print('AVISO: nenhuma GPU detectada. Mude o runtime antes de continuar.')

---
## 3 · Instalar dependências

In [ ]:
!pip install -q "pandas==2.2.2" wfdb scipy
import tensorflow as tf, wfdb, numpy as np
print('TensorFlow :', tf.__version__)
print('wfdb       :', wfdb.__version__)
print('numpy      :', np.__version__)
print('GPUs visíveis:', tf.config.list_physical_devices('GPU'))

---
## 4 · Upload dos arquivos do projeto
Envie os **4 arquivos** da pasta :
- 
- 
- 
- 

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Arquivos enviados:', list(uploaded.keys()))

---
## 5 · Montar estrutura de pacotes
Cria a hierarquia  esperada pelos módulos.

In [ ]:
import shutil, pathlib

for folder in ['src', 'src/noise', 'src/models']:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)
    pathlib.Path(f'{folder}/__init__.py').touch()

moves = {
    'data_io.py'     : 'src/data_io.py',
    'contaminate.py' : 'src/noise/contaminate.py',
    'dataset.py'     : 'src/models/dataset.py',
    'model.py'       : 'src/models/model.py',
}
for src_file, dst in moves.items():
    if pathlib.Path(src_file).exists():
        shutil.copy(src_file, dst)
        print(f'  {src_file}  →  {dst}')
    else:
        print(f'  AVISO: {src_file} não encontrado — volte à célula 4.')

print('
Estrutura criada:')
!find src -name '*.py' | sort

---
## 6 · Baixar dados do PhysioNet
48 registros do MIT-BIH + 3 arquivos de ruído do NSTDB.

⏱ **Estimativa: 5–10 minutos.** Só precisa rodar uma vez por sessão.

In [ ]:
import sys
sys.path.insert(0, '/content')

from src.data_io import ensure_datasets, ALL_RECORDS, NOISE_RECORDS

print(f'Baixando {len(ALL_RECORDS)} registros MIT-BIH + {len(NOISE_RECORDS)} arquivos de ruído...')
ensure_datasets()
print('Dados prontos.')

---
## 7 · Sanity check do pipeline
Verifica janelamento, contaminação e geração de batches.
Esperado: .

In [ ]:
from src.models.dataset import (
    build_data_pools, make_train_dataset, make_val_dataset,
    TRAIN_RECORDS, TEST_RECORDS,
)

print('Construindo pools de dados...')
pools = build_data_pools()

print(f'
  janelas treino : {pools.ecg_train_windows.shape}')
print(f'  janelas val    : {pools.ecg_val_windows.shape}')
print(f'  janelas teste  : {pools.ecg_test_windows.shape}')

train_ds = make_train_dataset(pools, batch_size=8)
xb, yb = next(iter(train_ds))
print(f'
batch treino: {xb.shape}  {yb.shape}')
assert xb.shape == (8, 1024, 1), 'Forma inesperada!'
print('Sanity check OK.')

---
## 8 · Construir o modelo

In [ ]:
from src.models.model import build_model, build_callbacks

model = build_model()
model.summary(line_length=80)
print(f'
Parâmetros treináveis: {model.count_params():,}')

---
## 9 · Treinar
Checkpoints salvos no Drive a cada época que melhora a .
Se a sessão cair, use a célula **10**.

| Parâmetro | Valor | Motivo |
|---|---|---|
|  | 150 | EarlyStopping interrompe antes se convergir |
|  | 32 | Cabe na T4 com folga |
|  | 400 | ~12.800 amostras/época |
|  | 100 | ~3.200 amostras de validação/época |

In [ ]:
EPOCHS          = 150
BATCH_SIZE      = 32
STEPS_PER_EPOCH = 400
VAL_STEPS       = 100
CHECKPOINT_PATH = f'{DRIVE_DIR}/checkpoints/best_model.keras'
LOG_DIR         = f'{DRIVE_DIR}/logs'

train_ds = make_train_dataset(pools, batch_size=BATCH_SIZE, seed=42)
val_ds   = make_val_dataset(pools,   batch_size=BATCH_SIZE, seed=123)

callbacks = build_callbacks(CHECKPOINT_PATH)
callbacks[-1] = tf.keras.callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=0)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    verbose=1,
)

best_val = min(history.history['val_loss'])
best_ep  = history.history['val_loss'].index(best_val) + 1
print(f'
Melhor val_loss: {best_val:.6f}  (época {best_ep})')
print(f'Checkpoint salvo em: {CHECKPOINT_PATH}')

---
## 10 · Retomar treino (use se a sessão cair)
Rode as células 1, 3, 6 e 7 antes desta.

In [ ]:
CHECKPOINT_PATH = f'{DRIVE_DIR}/checkpoints/best_model.keras'

import tensorflow as tf
model_retomado = tf.keras.models.load_model(CHECKPOINT_PATH)
print('Modelo carregado:', CHECKPOINT_PATH)

from src.models.dataset import make_train_dataset, make_val_dataset
from src.models.model import build_callbacks

train_ds = make_train_dataset(pools, batch_size=32, seed=42)
val_ds   = make_val_dataset(pools,   batch_size=32, seed=123)
callbacks = build_callbacks(CHECKPOINT_PATH)
callbacks[-1] = tf.keras.callbacks.TensorBoard(
    log_dir=f'{DRIVE_DIR}/logs', histogram_freq=0
)

history2 = model_retomado.fit(
    train_ds,
    epochs=150,
    steps_per_epoch=400,
    validation_data=val_ds,
    validation_steps=100,
    callbacks=callbacks,
    verbose=1,
)

---
## 11 · Curvas de aprendizado

In [ ]:
import matplotlib.pyplot as plt

hist = history.history  # use history2 se retomou na célula 10

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist['loss'],     label='treino')
axes[0].plot(hist['val_loss'], label='validação')
axes[0].set_title('MAE (loss)')
axes[0].set_xlabel('Época')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist['mse'],     label='treino')
axes[1].plot(hist['val_mse'], label='validação')
axes[1].set_title('MSE')
axes[1].set_xlabel('Época')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = f'{DRIVE_DIR}/curvas_treino.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print('Gráfico salvo em:', plot_path)

---
## 12 · Visualizar filtragem de exemplo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.models.dataset import build_test_table

best_model = tf.keras.models.load_model(CHECKPOINT_PATH)
print('Gerando tabela de teste...')
test_table = build_test_table(pools)

NOISE_VIS = 'bw'  # bw | ma | em | 60hz | 50hz
SNR_VIS   = 6     # 0 | 6 | 12 | 18

X, Y = test_table[(NOISE_VIS, SNR_VIS)]
idx  = 0

x_in  = X[idx:idx+1]
y_hat = best_model.predict(x_in, verbose=0)
y_ref = Y[idx]

t = np.arange(1024) / 360

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(t, x_in[0, :, 0],  color='tab:orange', lw=0.8)
axes[0].set_title(f'Entrada ruidosa  ({NOISE_VIS}, SNR={SNR_VIS} dB)')
axes[1].plot(t, y_hat[0, :, 0], color='tab:blue',   lw=0.8)
axes[1].set_title('Saída do modelo (sinal filtrado)')
axes[2].plot(t, y_ref[:, 0],    color='tab:green',  lw=0.8)
axes[2].set_title('Referência limpa')

for ax in axes:
    ax.set_ylabel('mV (norm.)')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Tempo (s)')

plt.tight_layout()
plot_path = f'{DRIVE_DIR}/exemplo_filtragem_{NOISE_VIS}_{SNR_VIS}dB.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print('Gráfico salvo em:', plot_path)